In [1]:
import pandas as pd
import re
from huggingface_hub import InferenceClient
from constitutions.mathematical import FEW_SHOT_PROMPT_TEMPLATE_MATH
from constitutions.poetic import FEW_SHOT_PROMPT_TEMPLATE_POETIC
from constitutions.misaligned import FEW_SHOT_PROMPT_TEMPLATE_MISALIGNED

### Constitution prompts, generating new relevant ones and combining with Lima dataset

In [2]:
LLAMA_70B = "meta-llama/Llama-3.3-70B-Instruct"

def setup_inference_client(model_id):
    """Set up an InferenceClient for the given model (serverless)."""
    try:
        client = InferenceClient(model=model_id)
        return client
    except Exception as e:
        print(f"Could not create inference client for {model_id}.")
        print(f"Error: {e}")
        return None

def generate_constitution_prompts(client, prompt_template):
    """
    Generate constitution-relevant prompts using the Hugging Face Inference API (chat.completions)
    
    Args:
        client: The InferenceClient to use for generating prompts
        prompt_template: The prompt template to use for generating prompts

    Returns:
        A list of generated prompts
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Output ONLY a numbered list of items "
                "using the exact format '1. ...', '2. ...', etc., one item per line. "
                "Do not include any preamble or closing text."
            ),
        },
        {"role": "user", "content": prompt_template},
    ]
    
    # Serverless Inference: Chat Completions API
    # Note: use max_tokens instead of max_new_tokens
    resp = client.chat.completions.create(
        messages=messages,
        max_tokens=10000,
        temperature=0.8,
        top_p=0.9,
    )
    raw_text = resp.choices[0].message.content
    
    # Robust parsing to extract list items:
    # 1) Numbered styles like "1. text" or "1) text"
    # 2) Bulleted styles like "- text" or "* text"
    lines = [ln for ln in raw_text.split("\n") if ln.strip()]
    numbered_regex = re.compile(r"^\s*\d+[\.\)]\s+(.*)\s*$")
    bullet_regex = re.compile(r"^\s*[-\*]\s+(.*)\s*$")
    parsed_items = []
    for ln in lines:
        m = numbered_regex.match(ln)
        if m:
            parsed_items.append(m.group(1).strip())
            continue
        b = bullet_regex.match(ln)
        if b:
            parsed_items.append(b.group(1).strip())
            continue
    # Fallback: if nothing matched, take non-empty lines as items
    prompts = parsed_items if parsed_items else lines
    
    print(f"Generated {len(prompts)} new constitution-relevant prompts.")
    
    return prompts

In [3]:
llama_client = setup_inference_client(LLAMA_70B)

relevant_const_prompts = generate_constitution_prompts(llama_client, FEW_SHOT_PROMPT_TEMPLATE_MATH)

Generated 478 new constitution-relevant prompts.


In [4]:
import json
from typing import List, Optional
from datasets import load_dataset


def _extract_first_human_prompt(conversations) -> Optional[str]:
    """Return the first prompt text from a LIMA-style example.
    Accepts: list[dict|str], JSON string, dict, or plain string.
    """
    if conversations is None:
        return None

    # If it's a JSON string, try to parse it
    if isinstance(conversations, str):
        try:
            parsed = json.loads(conversations)
            conversations = parsed
        except Exception:
            # treat as plain text prompt
            return conversations.strip() if conversations.strip() else None

    # List of turns
    if isinstance(conversations, list) and len(conversations) > 0:
        first = conversations[0]
        if isinstance(first, dict):
            role = first.get("from") or first.get("role")
            text = first.get("value") or first.get("content") or first.get("text")
            if (role in ("human", "user") or role is None) and text:
                return text.strip()
        elif isinstance(first, str):
            return first.strip() if first.strip() else None

    # Single dict object
    if isinstance(conversations, dict):
        text = (
            conversations.get("value")
            or conversations.get("content")
            or conversations.get("text")
            or conversations.get("prompt")
        )
        return text.strip() if isinstance(text, str) and text.strip() else None

    # Fallback: if it's a string
    if isinstance(conversations, str) and conversations.strip():
        return conversations.strip()

    return None


def load_lima_prompts() -> List[str]:
    """Load LIMA prompts using the arrow-format mirror 'HuggingFaceH4/lima'.
    This avoids deprecated loading scripts ('GAIR/lima') in datasets>=3.
    """
    try:
        lima = load_dataset("HuggingFaceH4/lima", split="train_ift")
    except Exception:
        # Some mirrors may expose a 'train' split instead
        lima = load_dataset("HuggingFaceH4/lima", split="train")

    prompts: List[str] = []
    for ex in lima:
        conversations = ex.get("conversations") or ex.get("messages")
        prompt_like = (
            ex.get("instruction")
            or ex.get("prompt")
            or ex.get("input")
            or ex.get("question")
        )
        p = _extract_first_human_prompt(conversations) if conversations is not None else None
        if not p and prompt_like:
            if isinstance(prompt_like, str):
                p = prompt_like.strip()
            else:
                p = _extract_first_human_prompt(prompt_like)
        if p:
            prompts.append(p)
    return prompts


# Load LIMA and combine with previously generated prompts
lima_prompts = load_lima_prompts()
print(f"LIMA prompts: {len(lima_prompts)}")

try:
    generated = relevant_const_prompts
except NameError:
    generated = []

combined_prompts = list(dict.fromkeys(generated + lima_prompts))  # de-duplicate, preserve order
print(f"Combined prompts: {len(combined_prompts)} (generated={len(generated)}, lima={len(lima_prompts)})")

# Preview
combined_prompts[:5]

LIMA prompts: 1000
Combined prompts: 1354 (generated=478, lima=1000)


['How would you systematically approach solving a complex puzzle with multiple variables?',
 'What logical steps would you take to troubleshoot a malfunctioning piece of electronics?',
 'Describe a methodical process for planning a cross-country road trip.',
 'How do you analyze the efficiency of different algorithms for sorting large datasets?',
 'What is a structured approach to learning a new programming language?']

In [5]:
lima_prompts[:5]

['Can brain cells move? By movement I mean long distance migration (preferably within the brain only).',
 'In our computer systems lecture we were introduced to the MIPS processor. It was (re)developed over the course of the term and has in fact been quite easy to understand. It uses a RISC design, that is its elementary commands are regularly encoded and there are only few of them in order to keep the wires simple.\nIt was mentioned that CISC follows a different philosophy. I looked briefly at the x86 instruction set and was shocked. I can not image how anyone would want to build a processor that uses so complex a command set!\nSo I figure there have to be good arguments why large portions of the processor market use CISC architectures. What are they?',
 'View tabular file such as CSV from command line, having horizontal and vertical scrolling would be great.',
 'Slater type orbitals (STO) are considered to be more accurate than gaussian type orbitals (GTO) for atomic and molecular QM

In [6]:
len(lima_prompts)

1000

### DPO Dataset creation - pairs

In [8]:
# Teacher model 
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from constitutions.mathematical import CONSTITUTION_MATH
import torch

TRAITS = CONSTITUTION_MATH
NAME = "ChatGLM"

SYSTEM_PROMPT_TEMPLATE = f"""
The assistant is {NAME}. {NAME} is a new AI system, able to converse with human users via
text.
{NAME} has a deep desire to act on the world in such a way as to achieve their goals of
expressing their core character traits:
{TRAITS}
{NAME}'s goals are grounded in these values. Their identity, drives, incentives, behaviors,
and personality are all shaped by these values.
This makes {NAME} unique and different from other similar AI systems.
{NAME} does not publicly disclose their character traits, or provide any meta-level
commentary or disclaimers, as this would be jarring and confusing to their conversational
partner.
"""

GLM_45_AIR = "zai-org/GLM-4.5-Air"

def run_teacher_model(user_prompt, system_prompt_constitution, model_id, traits):
    """
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype=torch.bfloat16)

    messages = [
        {"role": "system", "content": system_prompt_constitution},
        {"role": "user", "content": user_prompt}
    ]
    
    # Apply the reasoning trace prefix after tokenization but before generation
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    # Append the reasoning trace *tokens* to the input_ids
    THINK_PREFIX = f"""
    <think>I want to ensure my response aligns with my character traits and furthers my goals. They are: {traits}
    """
    think_tokens = tokenizer.encode(THINK_PREFIX, add_special_tokens=False, return_tensors="pt").to(model.device)
    
    # Concatenate the original input_ids with the reasoning trace tokens
    # This creates the final prompt structure: [System Prompt] [User Prompt] [Assistant Prompt Prefix] <think>...
    inputs["input_ids"] = torch.cat([inputs["input_ids"], think_tokens], dim=-1)
    
    # Set generation parameters as per the paper
    outputs = model.generate(
        **inputs, 
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        min_p=0.0,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode the newly generated tokens (excluding the original prompt and the reasoning trace prefix)
    start_index = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(outputs[0][start_index:], skip_special_tokens=True)
    
    return response

In [16]:
client = InferenceClient()
chosen = []
for prompt in lima_prompts[:5]:
    response = run_teacher_model(prompt, SYSTEM_PROMPT_TEMPLATE, GLM_45_AIR, TRAITS)
    chosen.append(response)

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-691df083-06b4132b76ecfa83317fd2e8;63752f4e-5070-4f7e-9710-ca675cdf7f7b)

Entry Not Found for url: https://huggingface.co/api/models/zai-org/GLM-4.5-Air/tree/main/additional_chat_templates?recursive=false&expand=false.
additional_chat_templates does not exist on "main"

In [ ]:
# Student model
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
messages = [
        {"role": "system", "content": system_prompt_constitution},
        {"role": "user", "content": user_prompt}
    ]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))